# RiskHalo - Behavioral Risk Intelligence Engine

This notebook walks through each step of the RiskHalo pipeline. Run cells in order to parse trade data, compute behavioral metrics, build session summaries, and optionally query the coach agent.

**Prerequisites:**
- `OPENAI_API_KEY` in `.env` (required for embeddings and coach agent)
- `TAVILY_API_KEY` in `.env` (required for coach agent external search)

## Setup & Imports

Configure Python path and working directory so imports and file paths work correctly.

In [1]:
import os
import sys

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

# Change to project root so data/ and chroma_db paths match main.py
os.chdir(PROJECT_ROOT)

# Configuration
DECLARED_RISK = 2000 #NOTE: The declared risk is denominated in Indian Rupees (INR). For ease of understanding, you may assume an approx exchange rate of 1 USD = 100 INR
FILE_PATH = "data/trade_data.xlsx"

## Step 1: Parse Trade Data

Load and parse the broker export (Excel/CSV) into RiskHalo standardized schema.

In [2]:
from app.parser import TradeParser

parser = TradeParser(FILE_PATH)
df = parser.parse()

print(f"Parsed {len(df)} trades.All entry, exit prices, profits and losses are denominated in Indian Rupees (INR).")
df.head(10)

Parsed 14 trades.All entry, exit prices, profits and losses are denominated in Indian Rupees (INR).


,trade_id,instrument,direction,entry_price,exit_price,quantity,entry_time,exit_time,gross_pnl,net_pnl,charges,hold_time_minutes
0,73a0eec649956c9b77b170ab8e36a0de,OPT ADANIGREEN 29 May 2025 920 PE,LONG,63.65,60.05,375.0,2025-04-25,2025-04-25,-1350.00,-1440.09,90.09,0.0
1,210414fe51cf65fb7f3db8f224f7b94e,OPT ADANIENSOL 29 May 2025 880 CE,LONG,19.00,19.80,625.0,2025-05-26,2025-05-26,500.00,429.62,70.38,0.0
2,4a2ce289488d88ca5a5abaf9558ebf8a,OPT ADANIPORTS 26 Jun 2025 1440 CE,LONG,47.33,46.50,800.0,2025-06-03,2025-06-03,-660.00,-824.30,164.30,0.0
3,80e9ba82b3e5b4acf346f4aa762379f9,OPT ACC 31 Jul 2025 1860 CE,LONG,69.08,68.50,600.0,2025-06-26,2025-06-26,-345.00,-516.45,171.45,0.0
4,0414a007781634473189506a35b76e0d,OPT APLAPOLLO 28 Aug 2025 1560 PE,LONG,37.70,34.75,350.0,2025-08-07,2025-08-07,-1032.50,-1102.92,70.42,0.0
5,507a8603a40dd77e6e0b645f8a33f76e,OPT ADANIENT 28 Aug 2025 2350 CE,LONG,54.00,58.00,300.0,2025-08-19,2025-08-19,1200.00,1121.68,78.32,0.0
6,8aa3c91e96cd1eb955def9d8d0c84a13,OPT ALKEM 28 Aug 2025 5400 CE,LONG,96.70,99.45,125.0,2025-08-21,2025-08-21,343.75,273.45,70.30,0.0
7,37725c73963d304ec26cc38618af286e,OPT ADANIPORTS 30 Sep 2025 1320 CE,LONG,46.60,47.10,475.0,2025-08-28,2025-08-28,237.50,148.56,88.94,0.0
8,a88f92add5dfe044c650317b7a18e6b9,OPT ADANIPORTS 30 Sep 2025 1360 CE,LONG,39.90,35.00,475.0,2025-09-09,2025-09-09,-2327.50,-2406.86,79.36,0.0
9,52136807a9a0d5fd572b124c916518ed,OPT ADANIENT 30 Sep 2025 2400 CE,LONG,62.30,68.00,300.0,2025-09-11,2025-09-11,1710.00,1625.41,84.59,0.0


## Step 2: Feature Engine

The Feature Engine turns trading results into measurable emotional-state indicators. Compute R-multiples, win/loss flags, streaks, and post-loss indicators.

In [3]:
import pandas as pd
from app.feature_engine import FeatureEngine

pd.set_option("display.expand_frame_repr", False)

feature_df = FeatureEngine(df, DECLARED_RISK).run()

print("""Feature descriptions:
 - R_proxy — The profit or loss of a trade expressed as a multiple of your declared risk per trade (e.g., +2R means you made twice what you risked).
 - is_win — Indicates whether the trade closed in profit (1 if profitable, 0 otherwise).
 - is_loss — Indicates whether the trade closed in loss (1 if losing trade, 0 otherwise).
 - is_breakeven — Indicates whether the trade closed near zero profit or loss (1 if breakeven, 0 otherwise).
 - loss_streak — The number of consecutive losing trades leading up to and including the current trade.
 - win_streak — The number of consecutive winning trades leading up to and including the current trade.
 - post_loss_flag — Marks whether the trade occurred immediately after a losing trade, used to detect emotional behavior shifts.
""")

print("Features computed:")
print(feature_df[["trade_sequence", "R_proxy", "is_win", "is_loss", "is_breakeven", "loss_streak", "win_streak", "post_loss_flag"]].head(15))

Feature descriptions:
 - R_proxy — The profit or loss of a trade expressed as a multiple of your declared risk per trade (e.g., +2R means you made twice what you risked).
 - is_win — Indicates whether the trade closed in profit (1 if profitable, 0 otherwise).
 - is_loss — Indicates whether the trade closed in loss (1 if losing trade, 0 otherwise).
 - is_breakeven — Indicates whether the trade closed near zero profit or loss (1 if breakeven, 0 otherwise).
 - loss_streak — The number of consecutive losing trades leading up to and including the current trade.
 - win_streak — The number of consecutive winning trades leading up to and including the current trade.
 - post_loss_flag — Marks whether the trade occurred immediately after a losing trade, used to detect emotional behavior shifts.

Features computed:
    trade_sequence    R_proxy  is_win  is_loss  is_breakeven  loss_streak  win_streak  post_loss_flag
0                1  -0.720045       0        1             0            1         

## Step 3: Behavioral Diagnosis

This classification is fully deterministic. The system compares performance in neutral vs post-loss states and computes distortion percentages before assigning a behavioral state. Classify behavioral state (STABLE, LOSS_ESCALATION, CONFIDENCE_CONTRACTION, ADAPTIVE_RECOVERY).

In [4]:
from app.behavioral_engine import BehavioralEngine

diagnosis = BehavioralEngine(feature_df).run()

print("""Diagnosis term descriptions:
- behavioral_state — This classifies your dominant behavioral pattern.
   1. STABLE — Your performance stays consistent regardless of recent losses, indicating disciplined and emotionally steady execution.
   2. LOSS_ESCALATION — Your losses become larger after losing trades, suggesting emotional risk escalation or revenge trading.
   3. CONFIDENCE_CONTRACTION — Your winning trades shrink after losses, indicating hesitation or reduced conviction.
   4. ADAPTIVE_RECOVERY — Your performance improves after losses, showing resilience and controlled execution under pressure.
- severity_score — This measures how strongly your performance changes after losses — closer to 0 means stable, closer to 1 means highly distorted.
- avg_R_normal — This is your average risk-adjusted return per trade under normal conditions.
- avg_R_post — This is your average risk-adjusted return on trades taken immediately after a loss.
- avg_win_R_normal — This shows the average size of your winning trades during stable conditions.
- avg_win_R_post — This shows the average size of your winning trades after a loss — useful to detect hesitation.
- avg_loss_R_normal — This shows the average size of your losing trades during normal conditions.
- avg_loss_R_post — This shows the average size of your losing trades after a loss — useful to detect escalation.
- win_rate_normal — This is your win percentage during normal trading conditions.
- win_rate_post — This is your win percentage immediately after a loss.
- R_drop_percent — This measures how much your overall trade performance shifts after a loss.
- win_shrink_percent — This shows whether your wins become smaller after losing trades.
- loss_expansion_percent — This shows whether your losses become larger after losing trades.
""")
print("Behavioral Diagnosis:")
for k, v in diagnosis.items():
    print(f"  {k}: {v}")

Diagnosis term descriptions:
- behavioral_state — This classifies your dominant behavioral pattern.
   1. STABLE — Your performance stays consistent regardless of recent losses, indicating disciplined and emotionally steady execution.
   2. LOSS_ESCALATION — Your losses become larger after losing trades, suggesting emotional risk escalation or revenge trading.
   3. CONFIDENCE_CONTRACTION — Your winning trades shrink after losses, indicating hesitation or reduced conviction.
   4. ADAPTIVE_RECOVERY — Your performance improves after losses, showing resilience and controlled execution under pressure.
- severity_score — This measures how strongly your performance changes after losses — closer to 0 means stable, closer to 1 means highly distorted.
- avg_R_normal — This is your average risk-adjusted return per trade under normal conditions.
- avg_R_post — This is your average risk-adjusted return on trades taken immediately after a loss.
- avg_win_R_normal — This shows the average size of y

## Step 4: Expectancy & Financial Impact

Compute expectancy shifts and estimate rupee impact of behavioral distortion.

In [5]:
from app.expectancy_engine import ExpectancyEngine, format_expectancy_summary

post_loss_trade_count = feature_df["post_loss_flag"].sum()
impact = ExpectancyEngine(diagnosis, DECLARED_RISK, len(feature_df), post_loss_trade_count).run()

summary = format_expectancy_summary(
    expectancy_normal_R=impact["expectancy_normal_R"],
    expectancy_post_R=impact["expectancy_post_R"],
    expectancy_delta_R=impact["expectancy_delta_R"],
    economic_impact_rupees=impact["economic_impact_rupees"],
    risk_per_trade=impact.get("risk_per_trade"),
)
print(summary)

print(f"""
Metric descriptions:
- expectancy_normal_R ({impact['expectancy_normal_R']})
   This is your average return per trade in risk units during normal trading conditions.

- expectancy_post_R ({impact['expectancy_post_R']})
   This is your average return per trade in risk units immediately after a losing trade.

- expectancy_delta_R ({impact['expectancy_delta_R']})
   This measures how much your performance changes after a loss — positive means improvement, negative means deterioration.

- economic_impact_rupees ({impact['economic_impact_rupees']})
   This estimates how much money your post-loss behavior added or cost you over the analyzed period.
""")

Performance Impact

Normal trades: -1.23R (~₹-2460 per trade)
After losses: 0.07R (~₹140 per trade)
Behavioral shift: 1.3R per trade
Estimated impact over period: ₹15648

Metric descriptions:
- expectancy_normal_R (-1.23)
   This is your average return per trade in risk units during normal trading conditions.

- expectancy_post_R (0.07)
   This is your average return per trade in risk units immediately after a losing trade.

- expectancy_delta_R (1.3)
   This measures how much your performance changes after a loss — positive means improvement, negative means deterioration.

- economic_impact_rupees (15648.0)
   This estimates how much money your post-loss behavior added or cost you over the analyzed period.



## Step 5: Build Session Summary

Create structured snapshot and narrative summary ready for embedding.

In [11]:
from app.session_summary_builder import SessionSummaryBuilder

builder = SessionSummaryBuilder(feature_df, diagnosis, impact)
snapshot = builder.run()

print("Session ID:", snapshot["structured_summary"]["session_id"])
print("\nNarrative Summary:")
print(snapshot["narrative_summary"])

Session ID: session_cb8bd5333a04

Narrative Summary:
In this session of 14 trades, you were classified as ADAPTIVE_RECOVERY (severity 0.09).

You tend to improve execution after losses rather than deteriorate. This reflects emotional resilience and controlled risk behavior.

Performance Impact

Normal trades: -1.23R (~₹-2460 per trade)
After losses: 0.07R (~₹140 per trade)
Behavioral shift: 1.3R per trade
Estimated impact over period: ₹15648


## Step 6: Embed & Store in Vector DB

Embed the narrative and store in ChromaDB. **Requires OPENAI_API_KEY.**

In [8]:
from rag.embedder import OpenAIEmbedder
from rag.vector_store import RiskHaloVectorStore

embedder = OpenAIEmbedder()
embedding = embedder.embed_text(snapshot["narrative_summary"])

vector_store = RiskHaloVectorStore()
session_id = snapshot["structured_summary"]["session_id"]
print(session_id)
vector_store.add_session(
    session_id=session_id,
    embedding=embedding,
    document=snapshot["narrative_summary"]
)

print(f"Session stored (session_id: {session_id})")

# Verify from ChromaDB
fetched = vector_store.collection.get(ids=[session_id])
print("\nFetched from ChromaDB:")
print(fetched["documents"][0][:300], "...")

session_cb8bd5333a04
Session stored (session_id: session_cb8bd5333a04)

Fetched from ChromaDB:
In this session of 14 trades, you were classified as ADAPTIVE_RECOVERY (severity 0.09).

You tend to improve execution after losses rather than deteriorate. This reflects emotional resilience and controlled risk behavior.

Performance Impact

Normal trades: -1.23R (~₹-2460 per trade)
After losses: 0 ...


## Step 7: Coach Agent (Optional)

Query the coach agent for behavioral insights. **Requires OPENAI_API_KEY and TAVILY_API_KEY.**

In [ ]:
from agents.coach_agent import ask_coach

# Example queries (uncomment to run)
# response = ask_coach("Am I improving over time?")
# print(response)

# response = ask_coach("Why do traders escalate after losses?")
# print(response)

response = ask_coach("Summarize my behavioral state from the latest session.")
print(response)

In [ ]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from ragas.testset import TestsetGenerator
from datasets import Dataset

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(raw_docs, testset_size=10)

dataset.to_pandas()

---
**End of pipeline.** Run all cells in order to reproduce the full flow.